# Chapitre 5 - Exercices 5.6 à 5.10

Splines périodiques, dérivation des splines de lissage, schéma de calcul en base de
B-splines, forme de Reinsch, biais et variance d'une spline de lissage.

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline, make_interp_spline
from scipy.linalg import null_space, solveh_banded
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl.ch05_splines import (truncated_power_basis, bspline_basis, bspline_penalty_matrix,
                              natural_spline_basis, SmoothingSpline)
set_style()

## Exercice 5.6 : Splines périodiques

**Énoncé.** On veut ajuster une fonction périodique de période $T$ connue. Comment
modifier la base des puissances tronquées pour obtenir ce résultat ?

**Idée.** Une fonction $T$-périodique est déterminée par sa restriction à $[0, T)$. On
remplace donc $x$ par $\tilde x = x \bmod T \in [0, T)$ et l'on écrit, avec des nœuds
$0 < \xi_1 < \dots < \xi_K < T$,

$$
f(x) = \sum_{j=0}^{3}\beta_j\tilde x^j + \sum_{k=1}^{K}\theta_k(\tilde x - \xi_k)_+^3 .
$$

Cette fonction est périodique par construction, et $C^2$ sur $(0, T)$. Mais au point de
recollement $\tilde x = 0 \equiv T$, rien n'assure la régularité : il faut imposer
$f^{(r)}(T^-) = f^{(r)}(0^+)$ pour $r = 0, 1, 2$.

**Les trois contraintes.** En dérivant terme à terme ($\frac{d^r}{dx^r}x^j =
\frac{j!}{(j-r)!}x^{j-r}$ et de même pour les puissances tronquées) :

$$
\begin{aligned}
r = 0 :&\quad \beta_1T + \beta_2T^2 + \beta_3T^3 + \textstyle\sum_k\theta_k(T - \xi_k)^3 = 0,\\
r = 1 :&\quad 2\beta_2T + 3\beta_3T^2 + 3\textstyle\sum_k\theta_k(T - \xi_k)^2 = 0,\\
r = 2 :&\quad 6\beta_3T + 6\textstyle\sum_k\theta_k(T - \xi_k) = 0 .
\end{aligned}
$$

(À gauche de $\xi_1$, en $0^+$, seul le polynôme compte : $f(0^+) = \beta_0$,
$f'(0^+) = \beta_1$, $f''(0^+) = 2\beta_2$, ce qui a été soustrait.) Ce sont 3 contraintes
linéaires sur $K + 4$ coefficients : l'espace des splines cubiques périodiques est de
dimension $K + 1$ - autant que de nœuds sur le cercle ($0$ joue le rôle d'un nœud
supplémentaire, où seule la dérivée troisième peut sauter).

**Base explicite.** Si $\mathbf C$ est la matrice $3 \times (K + 4)$ des contraintes, une
base de son noyau $\mathbf V$ ($(K+4) \times (K+1)$) fournit la base périodique
$\mathbf H_{\text{pér}}(x) = \mathbf H(\tilde x)\mathbf V$. Implémentation :

In [2]:
def periodic_spline_basis(x, knots, T):
    """Base de splines cubiques T-périodiques (puissances tronquées + 3 contraintes)."""
    knots = np.asarray(knots, float)
    C = np.zeros((3, 4 + len(knots)))
    fact = [1, 1, 2, 6]
    for r in range(3):
        for j in range(r, 4):                    # x^j : dérivée r-ième en T moins en 0
            C[r, j] = fact[j] / fact[j - r] * T ** (j - r) - (fact[r] if j == r else 0)
        C[r, 4:] = fact[3] / fact[3 - r] * (T - knots) ** (3 - r)
    V = null_space(C)
    return truncated_power_basis(np.mod(x, T), knots) @ V

T = 1.0
kn = np.linspace(0, 1, 10)[1:-1]
print("dimension de la base périodique :", periodic_spline_basis(np.array([0.1]), kn, T).shape[1])

dimension de la base périodique : 9
